# 10 · Hyperparameter search

The train pipeline turns every hyperparameter written as a list into a search
space, runs one Ray Tune trial per configuration, keeps the best one, tests it
and writes everything to disk. This guide covers:

- the search-space syntax, and what the configuration accepts, rewrites or
  rejects;
- the search strategies (`grid`, `random`, `hopt`, `optuna`, `bohb`) and the
  schedulers that stop trials early (`fifo`, `asha`, `median`, `bohb`), next to
  per-trial early stopping;
- a closed-form sweep (EASE over `l2`) and an iterative one (BPR with Optuna
  and ASHA) in one run, and how to read what the writer produced;
- cross-validation: scoring each configuration as a mean over folds, and
  retraining the winner for the epochs the folds settled on;
- the optimizer, learning-rate scheduler, precision, gradient clipping and
  per-trial resources.

The sweeps are deliberately tiny (three EASE values, six BPR trials of at most
eight epochs on 16- or 32-dimensional embeddings, six cross-validation trials):
they show the mechanics, they do not tune anything.

**Requirements:** `pip install warprec jupyter`. Runs in five to ten minutes on
a laptop CPU, almost all of it Ray starting a worker process per trial. It
reads MovieLens-100K through `guide_data.movielens_100k()`, which downloads and
converts it exactly as guide 1 shows.

In [1]:
import contextlib
import json
import os
import re
import shutil
import sys
from pathlib import Path

# Ray and Optuna print their own lines (cluster start, package upload, new study,
# slow-step warnings) through handlers bound to the notebook when they are
# imported, so the redirection used further down cannot catch them. Keep Ray's
# errors and Optuna's warnings only.
os.environ["RAY_LOGGER_LEVEL"] = "error"

import optuna
import pandas as pd
import ray
import yaml
from IPython.utils.capture import capture_output
from loguru import logger
from pydantic import ValidationError

sys.path.insert(0, "..")
from guide_data import movielens_100k

optuna.logging.set_verbosity(optuna.logging.WARNING)
folder = movielens_100k()
RUNS = Path("../runs/10-hyperparameter-search")
RUNS.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", None)

## Search spaces

Every hyperparameter of a model is one of:

- a single value: fixed for every trial;
- a plain list: the values to try;
- a list whose first element names a search space, followed by its arguments:
  `[loguniform, 1e-4, 1e-2]`.

The model's configuration class validates the section and rewrites each value
into the third form. The pipeline does this for every entry under `models`,
looking the class up by the model's name, and so can you:

In [2]:
from warprec.utils.registry import params_registry

logger.disable("warprec")
bpr = {
    "embedding_size": [16, 32],
    "learning_rate": 0.001,
    "reg_weight": 0.0,
    "batch_size": 1024,
    "epochs": 8,
}
grid = params_registry.get("BPR", **bpr)
grid.model_dump(exclude={"meta", "optimization", "early_stopping"})

{'embedding_size': [<SearchSpace.GRID: 'grid'>, 16, 32],
 'reg_weight': [<SearchSpace.GRID: 'grid'>, 0.0],
 'batch_size': [<SearchSpace.GRID: 'grid'>, 1024],
 'epochs': [<SearchSpace.GRID: 'grid'>, 8],
 'learning_rate': [<SearchSpace.GRID: 'grid'>, 0.001]}

Under the default strategy, `grid`, every value becomes a grid: a single value
is a grid of one, a list is enumerated in full. Under any other strategy a plain
list becomes a `choice`, sampled from:

In [3]:
sampled = params_registry.get("BPR", **bpr, optimization={"strategy": "optuna"})
sampled.model_dump(exclude={"meta", "optimization", "early_stopping"})

{'embedding_size': ['choice', 16, 32],
 'reg_weight': ['choice', 0.0],
 'batch_size': ['choice', 1024],
 'epochs': ['choice', 8],
 'learning_rate': ['choice', 0.001]}

The named spaces are Ray Tune's: `choice`, `grid`, `uniform`, `quniform`,
`loguniform`, `qloguniform`, `randn`, `qrandn`, `randint`, `qrandint`,
`lograndint`, `qlograndint`. The `q` variants take a step after the bounds,
the `log` variants an optional base. The checks below run when the
configuration is loaded, before anything is trained; each row sets one BPR
hyperparameter under `strategy: optuna`.

In [4]:
def check(field: str, value: list, model: str = "BPR", **optimization) -> str:
    """What the configuration turns one hyperparameter into, or why it refuses it."""
    params = (bpr if model == "BPR" else {}) | {field: value}
    optimization.setdefault("strategy", "optuna")
    try:
        valid = params_registry.get(model, **params, optimization=optimization)
    except ValidationError as e:
        message = e.errors()[0]["msg"].removeprefix("Value error, ")
        return "rejected: " + message.split(" Received")[0].split(" Values received")[0]
    return str([getattr(v, "value", v) for v in getattr(valid, field)])


cases = {
    "loguniform": ("learning_rate", ["loguniform", 5e-4, 2e-2]),
    "qrandint": ("embedding_size", ["qrandint", 16, 64, 16]),
    "bounds reversed": ("learning_rate", ["loguniform", 1e-2, 1e-4]),
    "bounds off the step": ("embedding_size", ["qrandint", 10, 64, 16]),
    "float space on an int": ("embedding_size", ["uniform", 8, 64]),
    "misspelt space": ("learning_rate", ["unifrm", 1e-4, 1e-2]),
}
table = pd.DataFrame(
    [
        {
            "case": case,
            "field": field,
            "written": str(value),
            "becomes": check(field, value),
        }
        for case, (field, value) in cases.items()
    ]
    + [
        {
            "case": "out of the model's range",
            "field": "l2 (EASE)",
            "written": "[-1.0, 10.0]",
            "becomes": check("l2", [-1.0, 10.0], model="EASE"),
        }
    ]
).set_index("case")
table

,field,written,becomes
case,,,
loguniform,learning_rate,"['loguniform', 0.0005, 0.02]","['loguniform', 0.0005, 0.02]"
qrandint,embedding_size,"['qrandint', 16, 64, 16]","['qrandint', 16, 64, 16]"
bounds reversed,learning_rate,"['loguniform', 0.01, 0.0001]",rejected: The upper bound must be higher than the lower bound.
bounds off the step,embedding_size,"['qrandint', 10, 64, 16]",rejected: The upper and lower bound must be divisible by the quantization factor.
float space on an int,embedding_size,"['uniform', 8, 64]","['choice', 8, 64]"
misspelt space,learning_rate,"['unifrm', 0.0001, 0.01]","['choice', 'unifrm', 0.0001, 0.01]"
out of the model's range,l2 (EASE),"[-1.0, 10.0]",rejected: Values of l2 for EASE model must be > 0.


Three rows are easy to miss:

- a float distribution on an integer hyperparameter is not an error: it is
  replaced by a `choice` between the two bounds, with only a warning in the log;
- a misspelt space name is not caught either: it becomes the first option of a
  `choice`, and a trial that draws the string `'unifrm'` as its learning rate
  fails when it starts;
- each model also checks its own ranges (EASE's `l2` must be positive) on every
  value of the list.

A distribution needs a strategy that samples. Under `grid` it is rejected, with
an error about mixed types, because the first element is a string and the rest
are numbers. This configuration leaves the strategy at its default:

In [5]:
print(Path("configs/bad-space.yml").read_text())

# A search space the default grid strategy cannot enumerate: uniform is a
# distribution, and a grid has no number of points to take from it.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.1
writer:
  dataset_name: bad-space
  writing_method: local
  local_experiment_path: ../runs/10-hyperparameter-search/
models:
  EASE:
    l2: [uniform, 10.0, 500.0]
evaluation:
  top_k: [10]
  metrics: [nDCG]



In [6]:
from warprec.utils.config.config import load_train_configuration

try:
    load_train_configuration("configs/bad-space.yml")
except ValidationError as e:
    print(e.errors()[0]["msg"])

Value error, For the Grid Search optimization, the field l2 must have values of the same type. Values received: ['uniform', 10.0, 500.0].


## Strategies

`optimization.strategy` picks the Ray Tune search algorithm, and
`optimization.num_samples` how many trials it runs.

| strategy | what proposes the next configuration | `num_samples` |
|---|---|---|
| `grid` (default) | the full cross product of the `grid` lists | how many times to repeat the whole grid; leave it at 1 |
| `random` | independent draws from each space | number of trials |
| `hopt` | HyperOpt's tree-structured Parzen estimator | number of trials |
| `optuna` | Optuna's TPE sampler | number of trials |
| `bohb` | BOHB's model-based sampler; needs `pip install "warprec[bohb]"` | number of trials |

`grid` and `random` are the same Ray searcher; what differs is the space a
plain list becomes (`grid` or `choice`). A `choice` or a distribution cannot be
written under `grid` (it is rejected, as above), so sampling always means
changing the strategy. `num_samples` above 1 under `grid` repeats every grid
point and is logged as a likely mistake.

`properties.seed` (default 42) seeds every searcher, so the same configuration
proposes the same first trials. It is also the seed each trial's model is
built with.

The Bayesian strategies start with `properties.n_startup_trials` random trials
before their model of the space guides the search; WarpRec ignores the key, with
a warning, under the other strategies. A searcher only learns from trials that
have *finished*: with six trials and room to run them all at once, all six would
be proposed before any score came back, and the search would be random. The
BPR sweep below sets `max_concurrent_trials: 1`, so that every proposal after
the warm-up sees all the trials before it and the run is reproducible; on a
real budget, trade some of that for parallelism.

## Schedulers and early stopping

A scheduler compares trials with each other and stops the weak ones; early
stopping watches one trial on its own. Both act on the validation metric
(`evaluation.validation_metric`) each time a trial reports it, at the end of an
epoch.

| `optimization.scheduler` | stops a trial when | required `properties` |
|---|---|---|
| `fifo` (default) | never: every trial runs all its epochs | none |
| `asha` | at a rung (`grace_period`, then × `reduction_factor`, up to `max_t`) it is not in the top 1/`reduction_factor` of the trials that have reached that rung | `max_t`, `grace_period`, `reduction_factor` |
| `median` | after `grace_period`, its best score is below the median of the running averages of the finished trials at the same point | `grace_period` |
| `bohb` | HyperBand pauses it; only the `bohb` strategy can resume it, so the two go together | `max_t`, `reduction_factor` |

The scheduler's clock is `properties.time_attr`, which defaults to
`completed_epochs`, so `max_t` and `grace_period` count epochs. A missing
required property fails validation:

In [7]:
check("l2", [100.0], model="EASE", scheduler="asha", properties={"grace_period": 2})

'rejected: Max_t property is required for ASHA scheduling. Change type of scheduling or provide the max_t attribute.'

`early_stopping` is per trial and works under any scheduler:

| key | meaning |
|---|---|
| `monitor` | `score` (default): the validation metric, at each evaluation, in the direction of `properties.mode`; `loss`: the epoch's mean training loss, `train_loss`, at every epoch, lower being better |
| `patience` | checks without improvement before the trial stops: evaluations under `score`, epochs under `loss` (required) |
| `grace_period` | the epoch from which the counting starts (default 0) |
| `min_delta` | how much better a value must be to count as an improvement (default 0) |

Every iterative model logs `train_loss`, so `monitor: loss` needs nothing from
the model. Closed-form models train in one step and report once, so neither a
scheduler nor early stopping has anything to act on.

## Running a sweep

One configuration holds both searches. EASE uses the default grid strategy over
three `l2` values and asks for its best model and recommendations to be saved
(`meta`). BPR draws six configurations with Optuna under ASHA, with early
stopping on the training loss. Its `optimization` block also asks for
`bf16-mixed` precision and gradients clipped to norm 1, as a configuration
written for a GPU cluster would; on a laptop CPU the trials fall back to full
precision and say so. Each model has its own `optimization` block, and the
models are searched one after the other.

In [8]:
print(Path("configs/sweep.yml").read_text())

# Two searches in one run on MovieLens-100K: each user's latest 10% is the test
# set, the latest 10% of what is left is the validation set every trial is
# scored on. Paths are relative to the guide's folder.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.1
  validation_splitting:
    strategy: temporal_holdout
    ratio: 0.1
writer:
  dataset_name: sweep
  writing_method: local
  local_experiment_path: ../runs/10-hyperparameter-search/
models:
  # Closed form: one trial per l2 value, grid search (the default strategy).
  EASE:
    meta:
      save_model: true
      save_recs: true
    optimization:
      num_workers: 0
    l2: [10.0, 100.0, 500.0]
  # Iterative: six configurations proposed by Optuna, pruned by ASHA and by
  # early stopping on the training loss. Sizes and epochs are kept tiny so the
  # guide r

`general.ray_address: local` starts a private Ray instance for the run instead
of connecting to a running cluster (the default, `auto`), so nothing has to be
started beforehand. Relative paths in the configuration are resolved from the
working directory, here the guide's folder. The pipeline does not shut that
instance down when it
returns, and a second pipeline call in the same process then fails with
`Maybe you called ray.init twice by accident?`; the helper below calls
`ray.shutdown()` after each run.

The helper also keeps the log out of the notebook. WarpRec logs through loguru
to standard output, and the trials log from their own processes, which Ray
forwards to this one; both go to `<dataset_name>.log` next to the output
folder. It deletes that folder first: run twice under the same configuration,
the train pipeline resumes the first run and skips the models it finished
(guide 11 covers runs and resuming).

In [9]:
from warprec.pipelines import train_pipeline

logger.remove()  # WarpRec's console output; each run logs to a file instead
logger.enable("warprec")
ANSI = re.compile(r"\x1b\[[0-9;]*m")


def train(config: str) -> Path:
    """Run the train pipeline from a clean output folder; return that folder."""
    name = yaml.safe_load(Path(config).read_text())["writer"]["dataset_name"]
    out = RUNS / name
    shutil.rmtree(out, ignore_errors=True)
    with (
        open(RUNS / f"{name}.log", "w") as log,
        contextlib.redirect_stdout(log),
        contextlib.redirect_stderr(log),
        capture_output(stdout=False, stderr=False),  # Ray Tune's empty status widget
    ):
        sink = logger.add(log, format="{message}", colorize=False)
        try:
            train_pipeline(config)
        finally:
            logger.remove(sink)
            ray.shutdown()
    return out


def log_lines(out: Path, *patterns: str) -> list[str]:
    """The lines of a run's log that contain one of the patterns, unprefixed."""
    lines = []
    for line in (RUNS / f"{out.name}.log").read_text().splitlines():
        line = ANSI.sub("", line)
        if any(p in line for p in patterns):
            line = re.sub(r"^\(\w+ pid=\d+\)\s*", "", line)  # Ray's process prefix
            lines.append(re.sub(r"^.* - WarpRec - \S+ ", "", line))
    return lines

In [10]:
sweep = train("configs/sweep.yml")
print("\n".join(log_lines(sweep, "Best params", "Early stopping triggered")))

Best params: {'l2': 100.0} | Score (nDCG@10): 0.13485734164714813 | Iteration: 1
Early stopping triggered at epoch 5 (Loss: 0.2046, Best: 0.2205).
Early stopping triggered at epoch 4 (Loss: 0.1880, Best: 0.2061).
Early stopping triggered at epoch 4 (Loss: 0.1431, Best: 0.1625).
Best params: {'embedding_size': 32, 'reg_weight': 0.0007319939418114051, 'batch_size': 1024, 'epochs': 8, 'learning_rate': 0.004550475813202187} | Score (nDCG@10): 0.1035303846001625 | Iteration: 6


The log names the best configuration of each model, its validation score and
the iteration it was reached at. For EASE the iteration is always 1. For BPR it
is the epoch whose report scored best, and that epoch's checkpoint, not
necessarily the last epoch, is the model that goes on to be tested. The
early-stopping lines come from inside the BPR trials, count epochs from 0 and
report the training loss they watched; the trials below show which ones.

Each BPR trial also logged the precision it was given, falling back on the CPU:

In [11]:
fallback = log_lines(sweep, "Mixed precision")
print(len(fallback), "trials logged:", fallback[0])

6 trials logged: Mixed precision 'bf16-mixed' was requested on the cpu, where it does not speed anything up. Falling back to '32-true'.


## What the writer produced

Everything lands under `writer.local_experiment_path/<dataset_name>/`:

In [12]:
for path in sorted(sweep.iterdir()):
    entries = sorted(p.name for p in path.iterdir())
    print(
        f"{path.name + '/':13}",
        entries if len(entries) < 4 else f"{len(entries)} entries",
    )

evaluation/   ['Overall_Results_08.10.2026_12-26-57.tsv', 'Time_Report_08.10.2026_12-26-57.tsv']
params/       ['Overall_Params_08.10.2026_12-26-57.json']
ray_results/  11 entries
recs/         ['EASE_08.10.2026_12-26-57.tsv']
run_state/    ['sweep_7ec0b7ae.json']
serialized/   ['EASE_PinkTrogonOfMastery.pth']
split/        []


| folder | holds |
|---|---|
| `evaluation/` | `Overall_Results_<timestamp>.tsv`, the test metrics of each model's best configuration, and `Time_Report_<timestamp>.tsv` |
| `params/` | `Overall_Params_<timestamp>.json`, each model's best hyperparameters and best iteration |
| `recs/` | the top-k lists of models with `meta.save_recs` |
| `serialized/` | the trained best model of models with `meta.save_model`, ready to serve (guide 17) |
| `ray_results/` | Ray Tune's record of every trial: one experiment folder per model, plus the trials' checkpoints |
| `run_state/` | what the pipeline needs to resume this run (guide 11) |
| `split/` | the splits, when `writer.save_split` is set (empty here) |

The timestamp is the run's start, so every file of one run carries the same
one, and each model adds its rows to the same table. These are test scores:

In [13]:
results = pd.read_csv(next((sweep / "evaluation").glob("Overall_Results_*")), sep="\t")
results

,Model,Top@k,Recall,nDCG
0,BPR,10,0.090085,0.096793
1,BPR,20,0.157452,0.116747
2,EASE,10,0.106698,0.117068
3,EASE,20,0.189809,0.142834


In [14]:
json.loads(next((sweep / "params").glob("Overall_Params_*")).read_text())

{'EASE': {'Best Params': {'l2': 100.0}, 'Best Training Iteration': 1},
 'BPR': {'Best Params': {'embedding_size': 32,
   'reg_weight': 0.0007319939418114051,
   'batch_size': 1024,
   'epochs': 8,
   'learning_rate': 0.004550475813202187},
  'Best Training Iteration': 6}}

The time report puts the cost of the search next to the size of the model.
`Hyperparameter Exploration Time` is the wall time of the whole search, and
`Average Trial Time` the mean of the trials' own clocks, which start when Ray
schedules them. On a laptop both are mostly Ray starting worker processes: even
EASE's trials, which fit in a single step, take tens of seconds each.

In [15]:
times = pd.read_csv(next((sweep / "evaluation").glob("Time_Report_*")), sep="\t")
times.set_index("Model Name")[
    [
        "Trainable Params (Best Model)",
        "Hyperparameter Exploration Time",
        "Average Trial Time (s)",
        "Evaluation Time",
    ]
].T

Model Name,BPR,EASE
Trainable Params (Best Model),81856,0
Hyperparameter Exploration Time,0:03:02.546537,0:00:39.849276
Average Trial Time (s),21.976769,27.307278
Evaluation Time,0:00:00.054082,0:00:00.110733


The saved EASE files: the recommendations are named after the model and the
run, the serialized model after a name generated from its hyperparameters.

In [16]:
print([p.name for p in (sweep / "serialized").iterdir()])
pd.read_csv(next((sweep / "recs").glob("EASE_*")), sep="\t").head(3)

['EASE_PinkTrogonOfMastery.pth']


,user_id,item_id,rating
0,1,100,0.658805
1,1,423,0.522864
2,1,154,0.443009


## Reading the trials

The summary files keep only the winner. Every trial is in `ray_results/`: one
experiment folder per model, named `<run name>__<model>`, holding one folder
per trial with its hyperparameters (`params.json`) and one row per report
(`progress.csv`). The EASE grid, one validation report per trial:

In [17]:
def trials(out: Path, model: str) -> pd.DataFrame:
    """One row per report of every trial of a model, with its hyperparameters."""
    experiment = next((out / "ray_results").glob(f"*__{model}"))
    frames = []
    for trial in experiment.glob(f"{model}_*"):
        progress = pd.read_csv(trial / "progress.csv")
        params = json.loads((trial / "params.json").read_text())
        frames.append(progress.assign(trial=trial.name.rsplit("_", 1)[1], **params))
    return pd.concat(frames, ignore_index=True)


ease = trials(sweep, "EASE")
ease.set_index("l2").sort_index()[["nDCG@10"]].T

l2,10.0,100.0,500.0
nDCG@10,0.120208,0.134857,0.126994


The BPR trials, one row each in the order they ran: the sampled
hyperparameters, how many epochs Ray Tune recorded, the best validation score
among them and the training loss of the last one. Every iterative model reports
`train_loss` with its scores.

In [18]:
bpr_trials = trials(sweep, "BPR")
summary = (
    bpr_trials.groupby("trial")
    .agg(
        started=("timestamp", "min"),
        embedding_size=("embedding_size", "first"),
        learning_rate=("learning_rate", "first"),
        reg_weight=("reg_weight", "first"),
        epochs_recorded=("completed_epochs", "max"),
        best_nDCG_at_10=("nDCG@10", "max"),
        last_train_loss=("train_loss", "last"),
    )
    .sort_values("started")
    .drop(columns="started")
)
summary.round(5)

,embedding_size,learning_rate,reg_weight,epochs_recorded,best_nDCG_at_10,last_train_loss
trial,,,,,,
7919e014,32,0.00455,0.00073,6.0,0.10353,0.20457
d4bf35ed,16,0.01221,0.00006,4.0,0.09780,0.20612
73d89d18,32,0.01790,0.00002,4.0,0.09990,0.16254
ecd5a06f,32,0.00085,0.00090,2.0,0.02210,0.69117
cda0a376,32,0.00203,0.00097,2.0,0.07793,0.62874
2eb1c94c,16,0.00410,0.00051,2.0,0.08144,0.45997


ASHA saves the scores it compared at each rung next to the experiment
(`asha_rungs.json`), which shows which trials reached its rungs at epochs 2, 4
and 8 (none reached the last one here):

In [19]:
experiment = next((sweep / "ray_results").glob("*__BPR"))
rungs = json.loads((experiment / "asha_rungs.json").read_text())
pd.DataFrame(rungs).rename(columns=lambda c: f"epoch {int(float(c))}").sort_index(
    axis=1
).round(4)

,epoch 2,epoch 4,epoch 8
7919e014,0.0838,0.0968,NaN
d4bf35ed,0.0925,0.0959,NaN
73d89d18,0.0932,0.0947,NaN
ecd5a06f,0.0221,NaN,NaN
cda0a376,0.0779,NaN,NaN
2eb1c94c,0.0814,NaN,NaN


The trials ran one at a time. The first had nothing to be compared with at
the rungs, and early stopping ended it after its sixth epoch: its training loss
fell by less than `min_delta` (0.02) from one epoch to the next, although its
validation score was still rising. The loss is what `monitor: loss` watches,
and it is not what the search ranks trials by. Each later trial was kept at a
rung only if it was in the top half of the trials that had reached that rung
before it, so two stopped at epoch 4 and three at epoch 2. The two stopped at
epoch 4 account for the other two early-stopping lines: Ray stops a trial
asynchronously, so each trained a fifth epoch past ASHA's decision, early
stopping fired at its end, and Ray Tune never recorded it. The first three
trials are Optuna's random warm-up (`n_startup_trials: 3`); the three it then
proposed from its model scored lower, which is about what three observations
can buy.

## Cross-validation

With `validation_splitting: k_fold_cross_validation`, the training part of the
test split is cut into folds and the fold index becomes one more grid
dimension: two learning rates on three folds are six BPR trials. A
configuration is scored by its mean over the folds. Each trial also has a best
epoch, the one whose report scored highest; `properties.desired_training_it`
summarises the best epochs of the winning configuration's folds into one count
(`median`, the default, `mean`, `min` or `max`), and the winner is retrained on
the whole training part for that many epochs, then tested. `meta.save_model`
keeps the retrained model, which the end of this section checks.

In [20]:
print(Path("configs/cross-validation.yml").read_text())

# BPR tuned by 3-fold cross-validation: the training part of the temporal test
# split is cut into three folds, every learning rate is trained and scored on
# each fold, and the one with the best mean is retrained on the whole training
# part, for the epochs the folds settled on, and tested.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.1
  validation_splitting:
    strategy: k_fold_cross_validation
    folds: 3
writer:
  dataset_name: cross-validation
  writing_method: local
  local_experiment_path: ../runs/10-hyperparameter-search/
models:
  BPR:
    meta:
      save_model: true
    optimization:
      num_workers: 0
      properties:
        desired_training_it: median
    learning_rate: [0.002, 0.02]
    embedding_size: 16
    reg_weight: 0.0001
    batch_size: 1024
    epochs: 8
evaluation:
  top_k: [10, 20

In [21]:
cv = train("configs/cross-validation.yml")
print("\n".join(log_lines(cv, "Best params")))

Best params: {'iterations': 5, 'embedding_size': 16, 'reg_weight': 0.0001, 'batch_size': 1024, 'epochs': 8, 'learning_rate': 0.02} | Avg Score: 0.36546313762664795 | Std: 0.0105145258255862 | Iterations: 5


Each trial's `params.json` holds its `fold` and its `progress.csv` one row per
epoch, so the pipeline's choice can be rebuilt: the best score and best epoch
of every trial,

In [22]:
reports = trials(cv, "BPR")
best = reports.loc[reports.groupby(["learning_rate", "fold"])["nDCG@10"].idxmax()]
best = best.set_index(["learning_rate", "fold"])[["nDCG@10", "completed_epochs"]]
best.round(5)

nDCG@10  completed_epochs
learning_rate fold                           
0.002         0     0.34206               8.0
              1     0.36669               8.0
              2     0.32768               8.0
0.020         0     0.37622               5.0
              1     0.36496               5.0
              2     0.35521               4.0

then, per learning rate, the mean score the configurations are ranked by and
the median best epoch:

In [23]:
best.groupby("learning_rate").agg(
    mean_score=("nDCG@10", "mean"),
    std=("nDCG@10", "std"),
    median_best_epoch=("completed_epochs", "median"),
).round(5)

,mean_score,std,median_best_epoch
learning_rate,,,
0.002,0.34548,0.01973,8.0
0.020,0.36546,0.01051,5.0


In [24]:
print(json.loads(next((cv / "params").glob("Overall_Params_*")).read_text()))
pd.read_csv(next((cv / "evaluation").glob("Overall_Results_*")), sep="\t")

{'BPR': {'Best Params': {'embedding_size': 16, 'reg_weight': 0.0001, 'batch_size': 1024, 'epochs': 8, 'learning_rate': 0.02}, 'Best Training Iteration': 5}}


,Model,Top@k,Recall,nDCG
0,BPR,10,0.100631,0.113211
1,BPR,20,0.168416,0.131290


The log's `Avg Score` is the mean of the 0.02 row, and `Iterations` that
row's median best epoch, which is also the `Best Training Iteration` written to
the parameters file; `epochs` in `Best Params` stays the search's upper bound.
The fold scores are far above the sweep's validation scores because
`k_fold_cross_validation` deals each user's interactions into the folds in
file order (guide 3): a fold validates on interactions from the whole of the
user's history, not on its end.

The winner is retrained on the whole training part, 90% of every user's
interactions, and tested. With a single validation split the pipeline does not
retrain: the tested model is the best trial's own, trained without the
validation interactions. The BPR test scores of the sweep and of this run
therefore differ in more than their hyperparameters.

The saved model shows how long the retraining ran. The cell below trains the
same model in this process, on the same training part, with the same seed and
loader, for the full `epochs`, and compares its weights after every epoch with
the saved ones:

In [25]:
import copy
import logging
import warnings

import lightning as L
import torch

import warprec.recommenders  # noqa: F401  (registers the models)
from warprec.common.initialize import initialize_datasets
from warprec.data.reader import ReaderFactory
from warprec.utils.callback import WarpRecCallback
from warprec.utils.registry import model_registry

logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", module="lightning")

config = load_train_configuration("configs/cross-validation.yml")
main, _, _ = initialize_datasets(
    ReaderFactory.get_reader(config), WarpRecCallback(), config
)
saved = torch.load(next((cv / "serialized").glob("BPR_*.pth")), weights_only=False)
model = model_registry.get(
    name="BPR",
    params=saved["params"],
    interactions=main.train_set,
    info=main.info(),
    seed=42,
)
weights = {}


class EveryEpoch(L.Callback):
    """Keep a copy of the weights at the end of every epoch."""

    def on_train_epoch_end(self, trainer, module):
        weights[trainer.current_epoch + 1] = copy.deepcopy(module.state_dict())


loader = model.get_dataloader(
    interactions=main.train_set,
    sessions=main.train_session,
    num_workers=0,
    pin_memory=False,
    persistent_workers=False,
)
L.Trainer(
    max_epochs=model.epochs,
    accelerator="cpu",
    logger=False,
    enable_checkpointing=False,
    enable_progress_bar=False,
    enable_model_summary=False,
    callbacks=[EveryEpoch()],
).fit(model, train_dataloaders=loader)
same = {
    epoch: all(torch.equal(saved["state_dict"][k], w[k]) for k in saved["state_dict"])
    for epoch, w in weights.items()
}
pd.Series(same, name="identical to the saved model").rename_axis("epochs").to_frame().T

epochs,1,2,3,4,5,6,7,8
identical to the saved model,False,False,False,False,True,False,False,False


The saved model is identical to the one trained here after its fifth epoch,
and to no other: the retraining ran the five epochs `desired_training_it`
settled on, not the eight `epochs` allows.

## How an iterative model is optimised

These keys live under `optimization` next to the search settings, but describe
how each trial trains rather than how trials are chosen.

**Optimizer and learning-rate scheduler.** `optimizer` names a PyTorch optimizer
(`Adam`, the default, `AdamW`, `SGD`, `RMSprop`, `Adagrad`) and `lr_scheduler` a
PyTorch scheduler, each with its keyword arguments under `params`. The model's
own `learning_rate` is always the optimizer's `lr`, and the scheduler steps once
per epoch. In a configuration:

```yaml
optimization:
  optimizer: {name: SGD, params: {momentum: 0.9}}
  lr_scheduler: {name: StepLR, params: {step_size: 4, gamma: 0.5}}
```

Through the API the same two blocks are set on a model before training, and
Lightning asks the model for them with `configure_optimizers()`:

In [26]:
from warprec.data import Dataset
from warprec.data.reader import LocalReader
from warprec.data.schema import SplitSpec
from warprec.data.splitting import Splitter
from warprec.utils.config import LRSchedulerConfig, OptimizerConfig

logger.disable("warprec")
data = LocalReader().read_tabular(local_path=str(folder / "ratings.tsv"), sep="\t")
train_part, _, test_part = Splitter().split_transaction(
    data, test=SplitSpec(strategy="temporal_holdout", ratio=0.1)
)
dataset = Dataset(train_data=train_part, eval_data=test_part, rating_type="implicit")
model = model_registry.get(
    name="BPR",
    params={**bpr, "embedding_size": 16},
    interactions=dataset.train_set,
    info=dataset.info(),
    seed=42,
)
model.set_optimization_parameters(
    optimizer_config=OptimizerConfig(name="SGD", params={"momentum": 0.9}),
    lr_scheduler_config=LRSchedulerConfig(
        name="StepLR", params={"step_size": 4, "gamma": 0.5}
    ),
)
setup = model.configure_optimizers()
optimizer, scheduler = setup["optimizer"], setup["lr_scheduler"]["scheduler"]
print(
    type(optimizer).__name__, optimizer.defaults["lr"], optimizer.defaults["momentum"]
)
print(
    type(scheduler).__name__,
    scheduler.step_size,
    scheduler.gamma,
    setup["lr_scheduler"]["interval"],
)

SGDWrapper 0.001 0.9
StepLRWrapper 4 0.5 epoch


Names are checked against what WarpRec registers, ignoring case, when the
configuration loads; the parameters are not checked until a trial builds the
optimizer.

In [27]:
try:
    OptimizerConfig(name="Adamax", params={})
except ValidationError as e:
    print(e.errors()[0]["msg"])

Value error, The optimizer name provided is not supported. These are the supported optimizers: ['ADAM', 'ADAMW', 'SGD', 'RMSPROP', 'ADAGRAD']


`ReduceLROnPlateau` steps on `train_loss`, the epoch's mean training loss,
rather than on the validation metric: the loss exists on every epoch of every
run, the retraining after cross-validation included, so the schedule a
retrained model follows is the one its trials followed. Lower being better, the
scheduler's `mode` must stay `min`:

In [28]:
try:
    LRSchedulerConfig(name="ReduceLROnPlateau", params={"mode": "max"})
except ValidationError as e:
    print(e.errors()[0]["msg"])

Value error, ReduceLROnPlateau steps on the training loss, which falls as the model improves, so its mode must be 'min'.


Here it halves the learning rate after any epoch whose loss is not at least
10% below the best so far (`threshold: 0.1`, relative by default), with no
patience. The cell trains BPR for eight epochs and records the learning rate
each epoch ran with and the loss it ended on:

In [29]:
plateau = model_registry.get(
    name="BPR",
    params={**bpr, "embedding_size": 16, "learning_rate": 0.02},
    interactions=dataset.train_set,
    info=dataset.info(),
    seed=42,
)
plateau.set_optimization_parameters(
    lr_scheduler_config=LRSchedulerConfig(
        name="ReduceLROnPlateau",
        params={"factor": 0.5, "patience": 0, "threshold": 0.1},
    )
)
epochs = []


class LearningRate(L.Callback):
    """Record the learning rate of each epoch and the loss it ended on."""

    def on_train_epoch_start(self, trainer, module):
        lr = trainer.optimizers[0].param_groups[0]["lr"]
        epochs.append({"epoch": trainer.current_epoch + 1, "lr": lr})

    def on_train_epoch_end(self, trainer, module):
        epochs[-1]["train_loss"] = trainer.callback_metrics["train_loss"].item()


L.Trainer(
    max_epochs=plateau.epochs,
    accelerator="cpu",
    logger=False,
    enable_checkpointing=False,
    enable_progress_bar=False,
    enable_model_summary=False,
    callbacks=[LearningRate()],
).fit(
    plateau,
    train_dataloaders=plateau.get_dataloader(
        interactions=dataset.train_set, sessions=dataset.train_session, num_workers=0
    ),
)
pd.DataFrame(epochs).set_index("epoch").T.round(4)

epoch,1,2,3,4,5,6,7,8
lr,0.0200,0.0200,0.0200,0.0200,0.0100,0.0100,0.0050,0.005
train_loss,0.4831,0.2501,0.2045,0.1848,0.1656,0.1556,0.1479,0.143


The loss fell by less than 10% in epochs 4 and 6, so the learning rate was
halved for epochs 5 and 7. In a configuration the same scheduler is
`lr_scheduler: {name: ReduceLROnPlateau, params: {factor: 0.5, patience: 0,
threshold: 0.1}}`, and it steps the same way in every trial.

**Precision and gradient clipping.** `precision` (`32-true`, the default,
`16-mixed`, `bf16-mixed`, `64-true`) and `gradient_clip`, with
`gradient_clip_algorithm` (`norm`, the default, or `value`), are handed to the
Lightning trainer. One function turns them into the trainer's arguments, and it
is where the mixed modes are dropped off a GPU:

In [30]:
from warprec.recommenders.trainer.runtime import lightning_runtime

mixed = params_registry.get(
    "BPR", **bpr, optimization={"precision": "16-mixed", "gradient_clip": 1.0}
)
sink = logger.add(sys.stdout, format="{message}", colorize=False)
logger.enable("warprec")
print("cpu: ", lightning_runtime(mixed.optimization, "cpu"))
print("gpu: ", lightning_runtime(mixed.optimization, "gpu"))
logger.remove(sink)

Mixed precision '16-mixed' was requested on the cpu, where it does not speed anything up. Falling back to '32-true'.


cpu:  {'precision': '32-true', 'gradient_clip_val': 1.0, 'gradient_clip_algorithm': 'norm'}
gpu:  {'precision': '16-mixed', 'gradient_clip_val': 1.0, 'gradient_clip_algorithm': 'norm'}


On the CPU a mixed mode falls back to `32-true` with a warning, since it buys
nothing there; on a GPU without bfloat16, `bf16-mixed` falls back to
`16-mixed`. Models that propagate over a sparse adjacency matrix (the graph,
knowledge-graph and multimodal graph families) raise under either mixed mode,
because PyTorch has no half-precision sparse kernels, and some attention-based
sequential models raise under `16-mixed` only. Closed-form models ignore both
keys. The same block reaches every trial of a search and the retraining after
cross-validation, which is where the sweep's trials logged the fallback above.

**Resources.** Each trial is a Ray Train run: one worker process, or one per GPU
when `gpu_per_trial` is an integer above 1.

| key | meaning |
|---|---|
| `cpu_per_trial` | CPUs reserved for each trial (default 1) |
| `gpu_per_trial` | GPUs per trial (default 0): a fraction in (0, 1] shares a GPU between trials, an integer above 1 trains one trial on that many GPUs with DDP |
| `max_concurrent_trials` | trials running at once; by default WarpRec divides what the Ray cluster has by what one trial needs, so with `cpu_per_trial: 1` a 12-core machine runs 11 at once |
| `custom_resources_per_trial`, `label_selector` | Ray logical resources and node labels, to keep trials off nodes that cannot hold them |
| `device` | `cpu` or `cuda` for this model, overriding `general.device` |
| `num_workers` | data-loader worker processes per trial; by default derived from the trial's CPUs. The guide's configurations set 0, because on 100,000 interactions starting workers costs more than it saves |

None of this needs a GPU to configure: on a cluster, `gpu_per_trial: 0.5` with
`device: cuda` runs two trials per GPU. Guide 11 covers pausing and resuming a
search, and dashboards that follow its trials.

## Summary

| key (under `models.<Name>`) | what it does |
|---|---|
| `<hyperparameter>` | a value, a list, or `[space, args...]` |
| `optimization.strategy`, `num_samples` | the search algorithm and how many trials it runs |
| `optimization.scheduler`, `properties.max_t`, `grace_period`, `reduction_factor`, `time_attr` | which trials are stopped early, on what clock |
| `optimization.properties.seed`, `n_startup_trials`, `mode` | searcher and model seed, Bayesian warm-up, `max` or `min` of the validation metric |
| `optimization.properties.desired_training_it` | how the folds' best iterations are summarised under cross-validation |
| `early_stopping.monitor`, `patience`, `grace_period`, `min_delta` | stop one trial whose validation metric or training loss stops improving |
| `optimization.optimizer`, `lr_scheduler` | the PyTorch optimizer and scheduler of an iterative model |
| `optimization.precision`, `gradient_clip`, `gradient_clip_algorithm` | trainer precision and clipping |
| `optimization.cpu_per_trial`, `gpu_per_trial`, `max_concurrent_trials`, `num_workers` | resources per trial and parallelism |
| `meta.save_model`, `meta.save_recs` | write the best model and its recommendations |
| `evaluation.validation_metric` | the metric every trial is scored and compared on |
| `splitter.validation_splitting` | a single validation split, or `k_fold_cross_validation` |
| `general.ray_address` | `local` for a private Ray, `auto` (default) or an address to join a cluster |

The [Models configuration](https://warprec.readthedocs.io/en/latest/configuration/models/)
page lists every `optimization` and `early_stopping` key, and the [Training
pipeline](https://warprec.readthedocs.io/en/latest/pipelines/training/) page the
pipeline's flows and outputs.